# Online Retail Data Analysis – Customer, Product, and Revenue Insights

This project analyzes the Online Retail dataset, which contains transactional data from an e-commerce business selling products to customers across multiple countries.

The analysis focuses on key commercial aspects such as revenue trends, top-performing products, customer purchasing behavior, and geographic distribution of sales.

Using Python and pandas, the dataset is cleaned and explored to uncover patterns that can support business decision-making.

The goal of this project is to identify the main drivers of revenue, highlight customer and product dynamics, and provide actionable insights from the data.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
df = pd.read_csv('online_retail_II.csv', delimiter = ',')

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


# DATA CLEANING

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


First we have to convert InvoiceDate to datetime format

In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

Then we create a new column named Revenue (Quantity * Price) 

In [6]:
df['Revenue'] = df['Quantity'] * df['Price']

Now let's check for missing data

In [7]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
Revenue             0
dtype: int64

A large number of transactions have missing Customer ID values, which are most likely guest accounts / non-registered buyers. 
I keep them for revenue & time-series analyses, but I also create a backup dataframe for a possible more detailed customer analysis later called df_customer.

In [8]:
df_customer = df[df['Customer ID'].notna()].copy()

We also have a smaller amount (4382) of missing rows in the Description column. Here I also create a backup dataframe for later if I need to use product name-based analysis or charts for example.

In [9]:
df_products = df[df["Description"].notna()].copy()

Now, let us check for duplicate values!

In [10]:
df.duplicated().sum()

np.int64(34335)

Thats a lot of duplicates, BUT, for this dataset, it's seems safe to drop them directly because duplicate rows in retail transaction data (same invoice, product, quantity, price, date, customer) have no realistic business explanation. A real repeated purchase would generate a new Invoice ID. So, we just simply drop them.

In [11]:
df.drop_duplicates(inplace = True)

In retail transaction data, negative quantities represent cancelled orders / returns - let's see if we have some of those here.

In [12]:
df[df['Quantity'] < 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.4


In [13]:
df[df['Quantity'] < 0].shape[0]

22496

This suggests that all rows with negative quantities does have an invoice number starting with the letter C. Let's see whether that assumption is true or false!

In [14]:
df[df['Quantity'] < 0]['Invoice'].astype(str).str.startswith('C').all()

np.False_

It's obviously False, so let's count the instances where this is not true - i.e. where negative Quantity isn't associated with an invoice number starting with the letter C:

In [15]:
df[(df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C'))]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,-0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,-0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,-0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,-0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,-0.0
...,...,...,...,...,...,...,...,...,...
1060794,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom,-0.0
1060796,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom,-0.0
1060797,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom,-0.0
1062371,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom,-0.0


Okay, so what we see here is zero Revenues, zero Prices and non-existing Customer IDs. Also, the number of rows affected are only 3393 out of ~1M rows, so it's a relatively small amount - it should be safe to drop them!

In [16]:
df = df[df['Quantity'] > 0]

Let us check for prices that are less than zero - if any of it exists at all:

In [17]:
df[df['Price'] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,-53594.36
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,-44031.79
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,-38925.87
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06


These are only 5 rows with bad debt adjustments — purely accounting entries, not real transactions. 
Safe to drop them all.

In [18]:
df = df[df['Price'] > 0]

Last but not least, we are changing the country name IERIE (it's the Irish name for Ireland — "Éire" in Irish Gaelic) to simply Ireland
(this was an issue that came across much later in the project, but since it's part of data preparation, I decided to put it here where it belongs)

In [19]:
df['Country'] = df['Country'].replace('EIRE', 'Ireland')

# DATA ANALYSIS

In [20]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


So, for starters, I will do four visuals, and then move on to the next level analysis:
* Monthly revenue trend
* Top 10 products by revenue
* Top 10 customers by revenue
* Revenue by country

As a visualization tool I will use Plotly.

## 1 ) MONTHLY REVENUE TREND

In [21]:
# First I resample the data. 'ME' stands for Month — it groups all transactions by month and sums the revenue.
monthly_revenue = df.resample('ME', on = 'InvoiceDate')['Revenue'].sum().reset_index()

# Then I visualize the monthly revenues
fig = px.line(monthly_revenue, x = 'InvoiceDate', y = 'Revenue', title = 'Monthly Revenue Trend', height = 600)

# Tweaking the chart
fig.update_layout(
    title_font_size = 24,
    title_font_color ='black',
    title_x = 0.5,                  # centers the title
    plot_bgcolor = 'white',         # chart background
    xaxis_title = 'Month',
    yaxis_title = 'Revenue ($)',
    font = dict(family = 'Arial', size = 12),   # global font
    hovermode = 'x unified'         # cleaner hover tooltip
)
fig.update_traces(
    line_width = 5, 
    line_color = 'navy'
)
fig.add_hline(y = monthly_revenue['Revenue'].mean(), line_dash = 'dash', line_color = 'gray', annotation_text = 'Average')

# Finally, showing the chart
fig.show()

## Key Observations

- **Strong seasonality** — revenue peaks towards the end of each year (November/December), which is expected for a retail business due to Christmas shopping. This pattern repeats in both 2010 and 2011.
- **November 2010 spike** — the most dramatic peak in the entire dataset, crossing 1.4M, significantly above the average line.
- **Post-holiday drop** — after each peak there's a sharp decline in early months (Feb-Mar), which is typical retail behaviour after the holiday season.
- **Most months fall below average** — the average line sits at ~0.8M, but the majority of months are below it, meaning the average is pulled up by the strong Q4 spikes.
- **The last data point drops sharply** — this is likely December 2011 being incomplete (the dataset probably cuts off mid-month), so it shouldn't be interpreted as a real decline.



## Business Implications

- **Helps with inventory planning** — stock up before Q4.
- **Identifies slow periods** (early year) where promotions could be targeted.
- **Confirms the business is seasonally driven** — important context for any further analysis.

# 2.) PRODUCT ANALYSIS

**TOP PRODUCTS / REVENUE:**
Measures which products generate the most income for the business. High revenue products are the primary drivers of profitability and deserve priority in inventory management, marketing, and stock security.

**TOP PRODUCTS / QUANTITY:**
Measures which products are sold in the highest volume. A product can rank highly here without appearing in the revenue chart — indicating a low-price, high-volume item. These products are important for logistics and supply chain planning, as they move the most stock.

## 2.1) TOP PRODUCTS / REVENUE

In [22]:
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending = False).head(10).reset_index()

fig2 = px.bar(top_products, x = 'Revenue', y = 'Description', orientation = 'h', title = 'Top 10 Products by Revenue', height = 600)

fig2.update_layout(
    title_font_size = 24,
    title_font_color= 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Revenue ($)',
    yaxis_title = 'Product',
    font = dict(family = 'Arial', size = 12),
    yaxis = dict(categoryorder = 'total ascending')
)

fig2.update_traces(marker_color = 'navy')

fig2.show()

Upon closer inspection, 'Manual', 'Postage' and 'Dotcom Postage' are not real products but shipping fees and manual adjustments. These should be excluded for an accurate product analysis.

In [23]:
keywords_to_remove = ['Manual', 'POSTAGE', 'DOTCOM']
top_products = df[~df['Description'].str.contains('|'.join(keywords_to_remove), case = False, na = False)]
top_products = top_products.groupby('Description')['Revenue'].sum().sort_values(ascending = False).head(10).reset_index()

fig2 = px.bar(top_products, x = 'Revenue', y = 'Description', orientation = 'h', title = 'Top 10 Products by Revenue', height = 600)

fig2.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Revenue ($)',
    yaxis_title = 'Product',
    font = dict(family = 'Arial', size = 12),
    yaxis = dict(categoryorder = 'total ascending')
)

fig2.update_traces(marker_color = 'navy')

fig2.show()

## Key Observations

- **REGENCY CAKESTAND 3 TIER is the clear top performer** — generating over 300k in revenue, significantly ahead of all other products. This is a high-value item worth prioritizing in inventory and marketing.
- **Seasonal products are well represented** — Party Bunting, Paper Chain Kit 50's Christmas, and Chilli Lights suggest strong seasonal demand, consistent with the Q4 spikes we saw in the revenue trend.
- **After the ninth position** (CHILLI LIGHTS) revenues converge and the gap between products narrows significantly. The top 9 products generate disproportionately higher revenue, making them the primary candidates for inventory prioritization, targeted promotions, and stock security.


## Business Implications

- **Double down on the top 3** — Cakestand, T-light Holder and Paper Craft Little Birdie drive the most revenue and deserve priority in stock management and promotions.
- **The seasonal products confirm the Q4 pattern** — these items should be heavily stocked before November.
- **Home décor is the core category** — any expansion in product range should consider staying within this niche where the customer base clearly shops.

## 2.2) TOP PRODUCTS / QUANTITY

In [24]:
# Filtering out "non-products" just like before
keywords_to_remove = ['Manual', 'POSTAGE', 'DOTCOM']

# Top products df for the chart
top_products_qty = df[~df['Description'].str.contains('|'.join(keywords_to_remove), case = False, na = False)]
top_products_qty = top_products_qty.groupby('Description')['Quantity'].sum().sort_values(ascending = False).head(10).reset_index()

# Assembling the chart itself
fig4 = px.bar(top_products_qty, x = 'Quantity', y = 'Description', orientation = 'h', title = 'Top 10 Products by Quantity Sold', height = 600)

fig4.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Quantity Sold',
    yaxis_title = 'Product',
    font = dict(family = 'Arial', size = 13),
    yaxis = dict(categoryorder = 'total ascending')
)

fig4.update_traces(marker_color = 'navy')

fig4.show()

## Key Observations

- **World War 2 Gliders is the top product by quantity** (~110k units) yet it did not appear in the top revenue chart at all — a classic example of a high-volume, low-price item. It sells in enormous quantities but generates relatively little revenue per unit.
- **White Hanging Heart T-Light Holder appears in both charts** — top by revenue and #2 by quantity, making it the most consistently strong product in the entire dataset. It is both widely sold and valuable.
- **Cake cases appear multiple times** (Pink Paisley, Fairy, Retrospot, Retro Spot, Skull) — these are clearly a high-volume consumable category, bought repeatedly in bulk. Their absence from the revenue chart confirms they are very low unit price items.
- **The two charts have very little overlap** — most products here are completely different from the revenue top 10, confirming the earlier point that volume and value tell very different stories.

## Business Implications

- **White Hanging Heart T-Light Holder is the star product** — it should be treated as the anchor of the product catalogue, prioritized in stock, marketing, and reordering.
- **World War 2 Gliders and cake cases are high-frequency consumables** — reliable volume drivers that need consistent stock availability but shouldn't be mistaken for high-margin products.

## 3.1) TOP CUSTOMER / REVENUE

In [25]:
# Creating df for top customers / rev
top_customers = df.groupby('Customer ID')['Revenue'].sum().sort_values(ascending = False).head(10).reset_index()
top_customers['Customer ID'] = top_customers['Customer ID'].astype(int).astype(str)

# Creating the chart
fig3 = px.bar(top_customers, x = 'Revenue', y = 'Customer ID', orientation = 'h', title = 'Top 10 Customers by Revenue', height = 600)

fig3.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Revenue ($)',
    yaxis_title = 'Customer ID',
    font = dict(family = 'Arial', size = 13),
    yaxis = dict(categoryorder = 'total ascending')
)

fig3.update_traces(marker_color = 'navy')

fig3.show()

## Key Observations

- **Customer 18102 and 14646 are outliers** — generating 580k and 540k respectively, nearly double the third-highest customer (14156 at ~300k). These two alone likely represent a disproportionate share of total revenue.
- **Sharp drop after position 2** — unlike the top products chart where the decline was gradual, here there is a clear gap between the top 2 and the rest, suggesting these are likely wholesale or B2B clients rather than regular retail customers.
- **Positions 5-10 are closely clustered** — revenues between ~140k-250k, no standout performers in this group.

## Business Implications

- **Customer 18102 and 14646 are high-risk dependencies** — if either of these customers churns, the revenue impact would be significant. Retention efforts should prioritize them.
- **The top 2 are almost certainly B2B clients** — their order volumes are too large for individual retail buyers. It's worth investigating whether they are resellers or wholesale partners.

In [26]:
# Investigating customer behaviour for Customer no. 18102
df[df['Customer ID'] == 18102.0][['Invoice', 'Description', 'Quantity', 'Price', 'Revenue']].head(20)

,Invoice,Description,Quantity,Price,Revenue
54,489438,DINOSAURS WRITING SET,28,0.98,27.44
55,489438,SET OF MEADOW FLOWER STICKERS,30,1.69,50.70
56,489438,CHARLIE AND LOLA CHARLOTTE BAG,30,1.15,34.50
57,489438,JUMBO BAG CHARLIE AND LOLA TOYS,30,2.00,60.00
58,489438,JUMBO BAG TOYS,60,1.30,78.00
59,489438,COUNTRY COTTAGE DOORSTOP GREEN,32,2.50,80.00
60,489438,GINGHAM HEART DOORSTOP RED,32,2.50,80.00
61,489438,CHARLIE+LOLA RED HOT WATER BOTTLE,56,3.00,168.00
62,489438,CHARLIE LOLA BLUE HOT WATER BOTTLE,56,3.00,168.00
63,489438,CHARLIE+LOLA PINK HOT WATER BOTTLE,60,1.90,114.00


## Based on the data, this customer shows clear B2B signals:

- **High quantities** — 280, 300, 560, 600 units per item, and even 576-600 on parasols/umbrellas
- **Wide product variety** — many different products within the same invoice
- **Multiple invoices** — we can already see at least 3 different invoice numbers (489438, 490059...)
- **Bulk pricing pattern** — items like parasols at £3.00 for 600 units is classic wholesale pricing

**Conclusion:**
Customer 18102 is almost certainly a wholesale/B2B client. The Charlie & Lola branded products in large quantities also suggest they may be a toy or gift reseller stocking up on a licensed product line.

In [27]:
# Investigating customer behaviour for Customer no.  14646
df[df['Customer ID'] ==  14646.0][['Invoice', 'Description', 'Quantity', 'Price', 'Revenue']].head(20)

,Invoice,Description,Quantity,Price,Revenue
6433,489889,FELTCRAFT DOLL ROSIE,96,2.55,244.80
6434,489889,RIBBON REEL LACE DESIGN,120,1.85,222.00
6435,489889,RIBBON REEL STRIPES DESIGN,120,1.45,174.00
6436,489889,FELTCRAFT 6 FLOWER FRIENDS,120,2.10,252.00
6437,489889,HEART FILIGREE DOVE SMALL,288,1.06,305.28
6438,489889,SET/5 RED SPOTTY LID GLASS BOWLS,32,2.55,81.60
6439,489889,FOLKART ZINC HEART CHRISTMAS DEC,288,0.72,207.36
6440,489889,HEART DECORATION PAINTED ZINC,288,0.55,158.40
6441,489889,ANGEL DECORATION PAINTED ZINC,288,0.55,158.40
6442,489889,FOOD CONTAINER SET 3 LOVE HEART,144,1.65,237.60


## This customer is even more clearly a wholesale client:
- **Very high quantities** — 120, 288, 480 units per item consistently throughout the order
- **Huge variety** — 20 completely different product types in a single invoice (489889)
- **Bulk pricing** — items like zinc heart decorations at £0.55 for 288 units, paper napkins at £0.64 for 96 units
- **The WHITE HANGING HEART T-LIGHT HOLDER** — 480 units for £1,224 — this was our #2 top product by revenue, and this single customer is buying it in massive quantities

**Conclusion:** Customer 14646 is definitively a wholesale/B2B client — likely a home décor or gift shop reseller. The product mix (zinc decorations, felt dolls, heart ornaments) is very consistent with a boutique gift shop buying stock in bulk.

## 3.2) NUMBER OF INVOICES / CUSTOMER

In [28]:
# Creating the df for top customers / invoices
invoices_per_customer = df.groupby('Customer ID')['Invoice'].nunique().sort_values(ascending=False).head(10).reset_index()
invoices_per_customer.columns = ['Customer ID', 'Number of Invoices']
invoices_per_customer['Customer ID'] = invoices_per_customer['Customer ID'].astype(int).astype(str)

# Creating the chart
fig5 = px.bar(invoices_per_customer, x = 'Number of Invoices', y = 'Customer ID', orientation = 'h', title = 'Top 10 Customers by No. of Invoices', height = 600)

fig5.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Number of Invoices',
    yaxis_title = 'Customer ID',
    font = dict(family = 'Arial', size = 13),
    yaxis = dict(categoryorder = 'total ascending')
)

fig5.update_traces(marker_color = 'navy')

fig5.show()

## Key Observations

- **Customer 14911 leads with 398 invoices** — nearly ordering every single day over a 2-year period. This is an exceptionally engaged customer.
- **14646 and 18102 appear here too** — our top revenue customers also place a high number of orders (150+), confirming they are not just one-off bulk buyers but consistent, recurring clients.
- **The top 2 (14911 and 12748) are not in the top revenue chart** — meaning they order frequently but in lower values per order, suggesting they are loyal retail customers rather than wholesalers.
- **The list is relatively evenly distributed** from position 3 onwards (200-210 invoices), unlike the revenue chart which had more outliers.

## Business Implications

- **Customer 14911 is the most loyal customer in the dataset** — nearly 400 orders over 2 years warrants a VIP retention strategy regardless of their revenue contribution.
- **Frequency and revenue tell different stories** — the top revenue customers and top frequency customers barely overlap, which confirms two distinct customer segments: high-value wholesalers and high-frequency loyal retailers. Both need different engagement strategies.

## 3.3) AVERAGE SPEND / CUSTOMER

In [29]:
# Creating a subset of the df 
avg_spend_per_customer = df.groupby('Customer ID')['Revenue'].mean().sort_values(ascending=False).head(10).reset_index()
avg_spend_per_customer.columns = ['Customer ID', 'Average Spend']
avg_spend_per_customer['Customer ID'] = avg_spend_per_customer['Customer ID'].astype(int).astype(str)

# Creating the chart
fig6 = px.bar(avg_spend_per_customer, x = 'Average Spend', y = 'Customer ID', orientation = 'h', title = 'Top 10 Customers by Average Spend', height = 600)

fig6.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Average Spend ($)',
    yaxis_title = 'Customer ID',
    font = dict(family = 'Arial', size = 13),
    yaxis = dict(categoryorder = 'total ascending')
)

fig6.update_traces(marker_color = 'navy')

fig6.show()

## Key Observations

- **Customer 16446 is a massive outlier** — with an average spend of ~55k per transaction, dwarfing every other customer on the list. This is almost certainly a single large wholesale order or a data anomaly worth investigating.
- **None of these customers appeared in the top revenue or top invoices charts** — suggesting these are infrequent buyers who place very large individual orders, rather than loyal or recurring clients.
- **Sharp drop after position 1** — the second highest customer (15098) averages ~13k, less than a quarter of 16446, reinforcing that the top entry is an outlier rather than part of a trend.
- **Positions 5-10 are tightly clustered** between 1k-3k average spend, showing a more typical high-value retail or small wholesale behaviour.

## Business Implications

- **Customer 16446 warrants immediate investigation** — an average spend of 55k per order is either a major wholesale account that isn't appearing in other charts due to few total orders, or a data quality issue.
- **High average spend + absence from frequency charts = one-time or rare bulk buyers** — these customers may benefit from targeted re-engagement campaigns to convert them into recurring clients.

## 4.) COUNTRY ANALYSIS

**TOTAL REVENUE BY COUNTRY:** Measures which countries generate the most income for the business. 
This reveals the geographic concentration of revenue and highlights the most commercially 
important markets.

**NUMBER OF CUSTOMERS BY COUNTRY:** Measures the size of the customer base in each country. 
A country can have many customers without ranking highly in revenue — indicating a large but 
low-spending market, which may represent an untapped growth opportunity.

**AVERAGE REVENUE PER CUSTOMER BY COUNTRY:** Measures how much each customer spends on average 
in a given country. Combined with customer count it tells a richer story — a small market with 
high average spend may be more valuable than a large market with low average spend.

### 4.1.1) TOTAL REVENUE / COUNTRY - Interactive Map

In [30]:
# Creating the subset for the chart
revenue_by_country = df.groupby('Country')['Revenue'].sum().reset_index()
revenue_by_country['Revenue Tier'] = pd.qcut(revenue_by_country['Revenue'], q = 5, 
                                              labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# Creating the chart itself - using numpy for quantile-based buckets instead of gradient
fig8 = px.choropleth(revenue_by_country, locations = 'Country', locationmode = 'country names',
                     color = 'Revenue Tier', title = 'Total Revenue by Country', height = 600, width = 1000,
                     color_discrete_sequence = ['#f7fbff', '#9ecae1', '#4292c6', '#2171b5', '#08306b'],
                     hover_name = 'Country',
                     hover_data = {'Revenue': ':,.0f'},
                     category_orders = {'Revenue Tier': ['Very Low', 'Low', 'Medium', 'High', 'Very High']})

fig8.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    font = dict(family = 'Arial', size = 13),
    geo = dict(
        showframe = True,
        showcoastlines = True,
        showcountries = True,
        countrycolor = 'grey',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'lightgrey',
        showocean = True,
        oceancolor = 'lightblue'
    )
)

fig8.show()

/var/folders/_b/k826g3z97s14by78qlrfhd6w0000gn/T/ipykernel_83026/3114184104.py:7: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig8 = px.choropleth(revenue_by_country, locations = 'Country', locationmode = 'country names',


### 4.1.2) TOP COUNTRIES BY REVENUE - Bar Chart

There are two charts here, since the UK is responsible for so much of the revenue it dwarfs any other country in comparison - so the second chart doesn't include the UK.

In [31]:
# Creating subset 'Top Countries'
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()

# Chart 1: All countries including UK
fig9 = px.bar(top_countries, x = 'Country', y = 'Revenue', title = 'Top 10 Countries by Revenue (incl. UK)', height = 600)

fig9.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Country',
    yaxis_title = 'Revenue ($)',
    font = dict(family = 'Arial', size = 13)
)

fig9.update_traces(marker_color = 'navy')
fig9.show()

# Chart 2: Excluding UK to compare international markets
top_countries_ex_uk = top_countries[top_countries['Country'] != 'United Kingdom']

fig10 = px.bar(top_countries_ex_uk, x = 'Country', y = 'Revenue', title = 'Top 9 International Markets by Revenue (excl. UK)', height = 600)

fig10.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Country',
    yaxis_title = 'Revenue ($)',
    font = dict(family = 'Arial', size = 13)
)

fig10.update_traces(marker_color = 'navy')
fig10.show()

## Key Observations

- **The business is overwhelmingly UK-dependent** — Ireland, the top international market at ~650k, is less than 4% of UK revenue. The entire international operation is fragile by comparison.
- **International reach is almost entirely Western European** — no North American or Asian presence despite Australia proving non-European demand exists.
- **Ireland and the Netherlands are clear outperformers** — understanding what drives their lead over similarly-sized economies like France and Germany could provide a replicable playbook.

## Business Implications

- **UK concentration is a strategic risk** — international diversification should be a priority.
- **Non-European markets are the biggest white space** — a structured push into the US or Asia could meaningfully shift the revenue mix.

### 4.2.1) NUMBER OF CUSTOMERS / COUNTRY - Interacive Map

In [32]:
customers_by_country = df_customer.groupby('Country')['Customer ID'].nunique().reset_index()
customers_by_country.columns = ['Country', 'Number of Customers']

all_labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

mask_low = customers_by_country['Number of Customers'] == 1
customers_by_country['Customer Tier'] = None
customers_by_country.loc[mask_low, 'Customer Tier'] = 'Very Low'

rest = customers_by_country.loc[~mask_low, 'Number of Customers']
customers_by_country.loc[~mask_low, 'Customer Tier'] = pd.qcut(
    rest, q=4,
    labels=['Low', 'Medium', 'High', 'Very High'],
    duplicates='drop'
)

fig11 = px.choropleth(customers_by_country, locations='Country', locationmode='country names',
                      color='Customer Tier', title='Number of Customers by Country', height=600,
                      color_discrete_map={
                          'Very Low':  '#f7fbff',
                          'Low':       '#9ecae1',
                          'Medium':    '#4292c6',
                          'High':      '#2171b5',
                          'Very High': '#08306b'
                      },
                      hover_name='Country',
                      hover_data={'Number of Customers': ':,.0f'},
                      category_orders={'Customer Tier': all_labels})

fig11.update_layout(
    title_font_size=24,
    title_font_color='black',
    title_x=0.5,
    font=dict(family='Arial', size=13),
    geo=dict(
        showframe=True,
        showcoastlines=True,
        showcountries=True,
        countrycolor='grey',
        projection_type='natural earth',
        showland=True,
        landcolor='lightgrey',
        showocean=True,
        oceancolor='lightblue'
    )
)
fig11.show()

/var/folders/_b/k826g3z97s14by78qlrfhd6w0000gn/T/ipykernel_83026/904400964.py:17: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig11 = px.choropleth(customers_by_country, locations='Country', locationmode='country names',


### 4.2.2) NUMBER OF CUSTOMERS / COUNTRY - Bar Chart


In [33]:
# Chart 1: Top 10 countries including UK
fig12 = px.bar(customers_by_country.sort_values('Number of Customers', ascending = False).head(10),
               x = 'Country', y = 'Number of Customers',
               title = 'Top 10 Countries by Number of Customers (incl. UK)',
               hover_name = 'Country',
               hover_data = {'Number of Customers': ':,.0f'},
               height = 600)

fig12.update_layout(
    title_font_size = 24,
    title_font_color = 'black',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Country',
    yaxis_title = 'Number of Customers',
    font = dict(family = 'Arial', size = 13),
    xaxis_tickangle = -45,
    yaxis = dict(gridcolor = 'lightgrey')
)
fig12.update_traces(marker_color = 'navy')
fig12.show()

# Chart 2: Top 10 countries excluding UK
customers_ex_uk = customers_by_country[customers_by_country['Country'] != 'United Kingdom'].sort_values('Number of Customers', ascending = False).head(10)

fig13 = px.bar(customers_ex_uk,
               x = 'Country', y = 'Number of Customers',
               title = 'Top 10 International Markets by Number of Customers (excl. UK)',
               hover_name = 'Country',
               hover_data = {'Number of Customers': ':,.0f'},
               height = 600)

fig13.update_layout(
    title_font_size = 24,
    title_font_color = 'navy',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Country',
    yaxis_title = 'Number of Customers',
    font = dict(family = 'Arial', size = 13),
    xaxis_tickangle = -45,
    yaxis = dict(gridcolor = 'lightgrey')
)
fig13.update_traces(marker_color = 'navy')
fig13.show()

## Key Observations
- **UK dominance is even more extreme on customers than revenue** — 5,410 customers vs Germany's 107 makes the international customer base look almost negligible by comparison.
- **Germany and France lead internationally but with a very shallow pool** — 107 and 95 customers respectively is a thin base to build sustainable revenue on, making those markets fragile despite their top positions.
- **Cross-referencing with the revenue chart reveals a striking anomaly** — Ireland and the Netherlands led international revenue, yet neither appears prominently here. This confirms they are driven by a handful of high-value customers rather than broad market penetration.

## Business Implications
- **Customer acquisition internationally should be the #1 priority** — the numbers are too small in every market to weather any churn.
- **Ireland and Netherlands' revenue should not be mistaken for market strength** — it's likely concentrated in very few accounts, making it high-risk despite the strong revenue figures.


### 4.3.1) AVERAGE REVENUE / CUSTOMER BY COUNTRY

In [34]:
# Creating the subset
avg_revenue_by_country = df.groupby('Country').agg(
    Total_Revenue = ('Revenue', 'sum'),
    Unique_Customers = ('Customer ID', 'nunique')
).reset_index()
avg_revenue_by_country['Avg Revenue per Customer'] = avg_revenue_by_country['Total_Revenue'] / avg_revenue_by_country['Unique_Customers']

# Bin into tiers - manually assign 'Very Low' if needed to avoid duplicate edges
all_labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

mask_low = avg_revenue_by_country['Avg Revenue per Customer'] == avg_revenue_by_country['Avg Revenue per Customer'].min()
avg_revenue_by_country['Revenue Tier'] = None
avg_revenue_by_country.loc[mask_low, 'Revenue Tier'] = 'Very Low'

rest = avg_revenue_by_country.loc[~mask_low, 'Avg Revenue per Customer']
avg_revenue_by_country.loc[~mask_low, 'Revenue Tier'] = pd.qcut(
    rest, q = 4,
    labels = ['Low', 'Medium', 'High', 'Very High'],
    duplicates = 'drop'
)

fig14 = px.choropleth(avg_revenue_by_country, locations = 'Country', locationmode = 'country names',
                      color = 'Revenue Tier', title = 'Average Revenue per Customer by Country', height = 600,
                      color_discrete_map = {
                          'Very Low':  '#f7fbff',
                          'Low':       '#9ecae1',
                          'Medium':    '#4292c6',
                          'High':      '#2171b5',
                          'Very High': '#08306b'
                      },
                      hover_name = 'Country',
                      hover_data = {'Avg Revenue per Customer': ':,.2f'},
                      category_orders = {'Revenue Tier': all_labels})

fig14.update_layout(
    title_font_size = 24,
    title_font_color = 'navy',
    title_x = 0.5,
    font = dict(family = 'Arial', size = 13),
    geo = dict(
        showframe = True,
        showcoastlines = True,
        showcountries = True,
        countrycolor = 'grey',
        projection_type = 'natural earth',
        showland = True,
        landcolor = 'lightgrey',
        showocean = True,
        oceancolor = 'lightblue'
    )
)
fig14.show()

/opt/anaconda3/envs/py_latest/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/var/folders/_b/k826g3z97s14by78qlrfhd6w0000gn/T/ipykernel_83026/3176900833.py:22: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig14 = px.choropleth(avg_revenue_by_country, locations = 'Country', locationmode = 'country names',


### 4.3.2) AVERAGE REVENUE / CUSTOMER BY COUNTRY

In [35]:
top10_avg_revenue = (df.groupby('Country')
                       .agg(Total_Revenue = ('Revenue', 'sum'),
                            Unique_Customers = ('Customer ID', 'nunique'))
                       .reset_index())
top10_avg_revenue = top10_avg_revenue[top10_avg_revenue['Unique_Customers'] > 0].copy()
top10_avg_revenue['Avg Revenue per Customer'] = top10_avg_revenue['Total_Revenue'] / top10_avg_revenue['Unique_Customers']
top10_avg_revenue = top10_avg_revenue.sort_values('Avg Revenue per Customer', ascending = False).head(10)

fig15 = px.bar(top10_avg_revenue,
               x = 'Country', y = 'Avg Revenue per Customer',
               title = 'Top 10 Countries by Average Revenue per Customer',
               hover_name = 'Country',
               hover_data = {'Avg Revenue per Customer': ':,.2f'},
               height = 600)

fig15.update_layout(
    title_font_size = 24,
    title_font_color = 'navy',
    title_x = 0.5,
    plot_bgcolor = 'white',
    xaxis_title = 'Country',
    yaxis_title = 'Avg Revenue per Customer ($)',
    font = dict(family = 'Arial', size = 13),
    xaxis_tickangle = -45,
    yaxis = dict(gridcolor = 'lightgrey')
)
fig15.update_traces(marker_color = 'navy')
fig15.show()

## Key Observations
- **Ireland's ~120k average is an extreme outlier** — nearly 5x Singapore and the Netherlands (~25k each), strongly suggesting it's driven by one or two very large wholesale accounts rather than genuine market depth.
- **The UK not cracking the top 10 here is telling** — with 5,410 customers, its average is naturally diluted, confirming the UK is a high-volume/moderate-value market while the top of this chart represents low-volume/high-value accounts.
- **Germany and France, which led on customer count, are entirely absent here** — their large customer bases come with relatively low spend per head, the opposite profile to Ireland or Singapore.
## Business Implications
- **Ireland and Singapore's high averages should be investigated before being celebrated** — if they're driven by a single account, they represent concentration risk, not market success.
- **The contrast across all three charts points to a clear strategic choice** — pursue volume (Germany / France model) or pursue value (Ireland/Singapore model). Currently the business has no market where it achieves both simultaneously, which is the real gap.

## 5.) RFM ANALYSIS

RFM (Recency, Frequency, Monetary) is a customer segmentation technique that scores each customer 
across three dimensions:
- **Recency** — how recently did they purchase? (lower = better)
- **Frequency** — how many orders did they place?
- **Monetary** — how much revenue did they generate?

Each dimension is scored 1–5 using quintiles, combined into a single RFM score, 
and mapped to a customer segment.

In [36]:
df_customer_clean = df[df['Customer ID'].notna()].copy()

In [37]:
# Calculate RFM

# Snapshot date: one day after the last transaction in the dataset
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate R, F, M per customer
rfm = df_customer_clean.groupby('Customer ID').agg( ###
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary  = ('Revenue', 'sum')
).reset_index()

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [38]:
# Score into quintiles:

# Score Recency 5→1 (lower recency = better = higher score)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1]).astype(int)

# Score Frequency and Monetary 1→5 (higher = better)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1,2,3,4,5]).astype(int)

# Combine into RFM score string (e.g. '555')
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,326,12,77556.46,2,5,5,255
1,12347.0,2,8,4921.53,5,4,5,545
2,12348.0,75,5,2019.40,3,4,4,344
3,12349.0,19,4,4428.69,5,3,5,535
4,12350.0,310,1,334.40,2,1,2,212


In [39]:
# Map to segments:

def assign_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']     
    if r >= 4 and f >= 4 and m >= 4:
        return 'Best Customers'
    elif r == 1 and f >= 4 and m >= 4:
        return 'Cannot Lose Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f >= 2:
        return 'Potential Loyal Customers'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk Customers'
    elif r <= 2 and f <= 2:
        return 'Lost Customers'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

rfm['Segment'].value_counts()

Segment
Lost Customers               1523
Best Customers               1297
Loyal Customers              1138
At Risk Customers             556
Potential Loyal Customers     461
New Customers                 443
Needs Attention               400
Cannot Lose Customers          60
Name: count, dtype: int64

In [40]:
# Visualize segment distribution:

segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Count']

fig16 = px.bar(segment_counts.sort_values('Count', ascending=True),
               x='Count', y='Segment', orientation='h',
               title='Customer Segments (RFM)',
               height=500)

fig16.update_layout(
    title_font_size=24,
    title_font_color='navy',
    title_x=0.5,
    plot_bgcolor='white',
    xaxis_title='Number of Customers',
    yaxis_title='Segment',
    font=dict(family='Arial', size=13),
    xaxis=dict(gridcolor='lightgrey')
)

fig16.update_traces(marker_color='navy')
fig16.show()

In [41]:
# Segment revenue summary:

segment_summary = rfm.groupby('Segment').agg(
    Customers = ('Customer ID', 'count'),
    Avg_Recency   = ('Recency', 'mean'),
    Avg_Frequency = ('Frequency', 'mean'),
    Avg_Monetary  = ('Monetary', 'mean')
).round(1).reset_index().sort_values('Avg_Monetary', ascending=False)

segment_summary

,Segment,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary
1,Best Customers,1297,20.0,17.1,9143.9
2,Cannot Lose Customers,60,486.6,8.7,4115.1
0,At Risk Customers,556,345.8,5.3,2263.9
4,Loyal Customers,1138,73.4,5.9,2255.4
6,New Customers,443,28.1,1.5,885.5
7,Potential Loyal Customers,461,81.8,2.6,499.1
3,Lost Customers,1523,459.3,1.3,429.7
5,Needs Attention,400,257.3,2.0,415.3


## Key Observations
- **Best Customers are the engine** — 1,306 customers generating ~960 avg revenue with purchases just 19 days ago on average. This is a healthy, active core.
- **The Loyal Customer segment is surprisingly weak** — only 64 customers with high frequency and monetary but a 187-day recency gap. These were valuable customers who have quietly gone quiet and are dangerously close to At Risk.
- **1,535 Lost customers is the biggest segment by far** — averaging 466 days since last purchase, they are almost certainly gone. The low monetary average (340) suggests they were never high-value to begin with.
- **New Customers (459) look promising but fragile** — recent purchases but very low frequency (1.6) and low monetary (520). The critical question is whether they convert to Potential Loyalists or disappear.
- **At Risk Customers (396) is the most urgent segment** — these customers used to buy frequently (4.4 invoices) and spent reasonably well (1,182) but haven't returned in 367 days. They are reachable but the window is closing.

## Business Implications
- **Protect Best Customers at all costs** — 1,306 customers averaging ~960 revenue is the backbone of the business. Any churn here is disproportionately damaging.
- **Loyal Customers (64) needs immediate re-engagement** — the combination of high historical value and 187-day silence is a red flag. A targeted win-back campaign here has the highest ROI potential in the entire dataset.
- **At Risk Customers (396) is the second priority** — similar profile to Loyal but more customers. A time-sensitive promotion or personal outreach could recover a meaningful chunk of revenue.
- **Don't over-invest in Lost Customers** — 1,535 customers sounds alarming but their low average spend suggests they were likely one-time seasonal buyers. Recovery cost would likely outweigh the return.

## 6.) COHORT / RETENTION ANALYSIS

A cohort is a group of customers who made their first purchase in the same month.
By tracking how many of them return in subsequent months, we can measure customer retention
and understand the long-term value of each acquisition wave.

In [45]:
# Build the cohort table:

# Extract invoice month and first purchase month per customer
df_customer_clean['InvoiceMonth'] = df_customer_clean['InvoiceDate'].dt.to_period('M')

cohort_data = df_customer_clean.groupby('Customer ID')['InvoiceMonth'].min().reset_index()
cohort_data.columns = ['Customer ID', 'CohortMonth']

df_cohort = df_customer_clean.merge(cohort_data, on='Customer ID')

# Calculate months since first purchase
df_cohort['MonthsElapsed'] = (df_cohort['InvoiceMonth'] - df_cohort['CohortMonth']).apply(lambda x: x.n)

# Count unique customers per cohort/month combination
cohort_counts = df_cohort.groupby(['CohortMonth', 'MonthsElapsed'])['Customer ID'].nunique().reset_index()
cohort_pivot = cohort_counts.pivot(index='CohortMonth', columns='MonthsElapsed', values='Customer ID')

# Convert to retention %
cohort_retention = cohort_pivot.divide(cohort_pivot[0], axis=0).round(3) * 100
cohort_retention.index = cohort_retention.index.astype(str)

In [46]:
# Visualize a heatmap:

import plotly.graph_objects as go

fig17 = go.Figure(data=go.Heatmap(
    z=cohort_retention.values,
    x=[f'Month {i}' for i in cohort_retention.columns],
    y=cohort_retention.index,
    colorscale='Blues',
    text=cohort_retention.values.round(1),
    texttemplate='%{text}%',
    showscale=True
))

fig17.update_layout(
    title='Customer Retention by Cohort (%)',
    title_font_size=24,
    title_font_color='navy',
    title_x=0.5,
    font=dict(family='Arial', size=11),
    xaxis_title='Months Since First Purchase',
    yaxis_title='Cohort Month',
    height=600
)
fig17.show()

## Key Observations
- **Retention drops sharply after Month 0** — most cohorts lose 60-80% of customers after their first purchase. This is the single most important finding: the business is much better at acquiring customers than keeping them.
- **Retention stabilizes at roughly 10-25%** from Month 2 onwards for most cohorts — meaning only about 1 in 5 customers becomes a repeat buyer, and those who do tend to stick around.
- **The Jan 2010 cohort is a clear outlier** — showing ~49.6% retention at Month 11, nearly double any other cohort at the same stage. This aligns with the Christmas 2010 revenue spike we saw in the monthly trend — this cohort was likely acquired during a strong seasonal push and responded well to it.
- **NaN values for recent cohorts are expected** — a cohort from Oct 2011 simply hasn't had 12+ months to show retention data yet, so these are not missing data issues.
- **No cohort shows meaningful improvement over time** — retention curves flatten rather than recover, suggesting no re-engagement strategy is currently working at scale.

## Business Implications
- **First-to-second purchase conversion is the biggest lever in the business** — if even 10% more customers made a second purchase, the revenue impact would be significant across every cohort.
- **The Jan 2010 cohort behaviour should be studied** — whatever drove that unusually high Month 11 retention is worth identifying and replicating.
- **This directly validates the RFM findings** — the large Lost segment (1,535 customers) and weak Loyal segment (64 customers) are exactly what a retention curve this steep would produce.

# FINAL CONCLUSIONS

## Key Findings

- **Revenue is heavily seasonal** — the business generates a disproportionate share of its income in Q4, with November consistently the peak month. Most of the year operates below average.
- **A small number of products drives the majority of revenue** — the top 3 products alone account for an outsized share of sales, and they are concentrated in a single category: home décor and gifting.
- **Revenue is highly concentrated in a small number of customers** — the top 2 customers alone (identified as wholesale/B2B clients) generate over 1.1M combined, making the business vulnerable to individual account churn.
- **Retention drops sharply after the first purchase** — 60–80% of customers do not return after Month 0. Only ~1 in 5 converts into a repeat buyer, which directly explains the large Lost segment in the RFM analysis.
- **Geographic concentration is a significant risk** — over 90% of revenue originates from the UK. Every international market, including the top performer Ireland, generates less than 4% of UK revenue individually.

## Recommendations

- **Invest in first-to-second purchase conversion** — given the steep retention drop, even a modest improvement here would have a compounding effect across all cohorts and represents the highest-leverage opportunity in the business.
- **Build a formal key account programme for top B2B customers** — the top 2 customers represent an outsized share of revenue with no apparent retention strategy. Losing either would be immediately material.
- **Diversify internationally before it becomes urgent** — the UK concentration is not a problem until it is, at which point it is too late. Germany and France have the largest international customer bases and are the most logical markets to invest in first.